# Proyecto Fase 2. Ajuste del router de intención (mDeBERTa-v3) con mensajes de CAB

**Materia:** Gestión de proyectos de inteligencia artificial<br>
**Estudiante:** T07175241 - Gabriel Alejandro García Ángeles<br>
**Componente:** router de intención del chatbot de estatus de órdenes y ETA de Componentes Automotrices del Bajío (CAB)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gagarang25/ProyectoFase2/blob/main/entrenamiento/router_mdeberta_colab.ipynb)

En la Actividad 4 se eligió **mDeBERTa-v3** (`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`) como router del chatbot: el modelo que lee el mensaje del cliente y decide si se responde en automático (estatus o fecha de entrega) o si pasa a un ejecutivo. Ese ajuste se hizo solo con mensajes en inglés de Bitext. En la Actividad 5 una prueba con 50 mensajes escritos a mano mostró que el router no generalizaba bien, sobre todo en español, y la retroalimentación pidió **revalidar el modelo con consultas de CAB en español y manejar las solicitudes ambiguas antes de integrarlo en la API**.

Este cuaderno hace ese paso, que es el que se integra en el contenedor de la Fase 2:

1. Ajusta mDeBERTa-v3 con **data-v3.0.0**: Bitext (A4) más 2,000 mensajes simulados de CAB, 80 % en español.
2. Calibra la confianza del modelo y elige el **umbral** con el que la API decide entre responder, preguntar o escalar.
3. Evalúa con una **prueba externa de 160 mensajes escritos a mano**, que incluye español coloquial, *spanglish*, mensajes ambiguos y fuera de tema.
4. Publica el modelo en Hugging Face Hub con la etiqueta `model-v1.0.0`, para que el `Dockerfile` descargue siempre la misma versión.

El código está en `entrenamiento/entrenar_router.py` (repositorio del proyecto); el cuaderno lo ejecuta y muestra los resultados.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**. En *Secrets* (ícono de llave 🔑) debe existir `HF_TOKEN` con permiso de **escritura** y acceso activado para este cuaderno.

## 1. Configuración de GPU

El ajuste de un modelo de 278 millones de parámetros tarda unos minutos en GPU y horas en CPU. Esta celda comprueba que Colab asignó una GPU.

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("¿GPU disponible?:", torch.cuda.is_available())

## 2. Código y datos desde el repositorio

Se clona el repositorio de la Fase 2. Entrenar desde un commit ya publicado permite que el modelo quede ligado a una versión verificable del código y de los datos (retroalimentación de la Actividad 5): el hash del commit se guarda dentro del modelo, en `config_cab.json`.

In [ ]:
import os
REPO_GITHUB = "https://github.com/gagarang25/ProyectoFase2.git"
if not os.path.exists("/content/ProyectoFase2"):
    !git clone -q {REPO_GITHUB} /content/ProyectoFase2
%cd /content/ProyectoFase2
!git log -1 --format="Commit %H%nFecha  %cd%nMensaje %s"

In [ ]:
# Comprobación de integridad: las huellas SHA-256 deben coincidir con datos/mensajes_cab/versiones.json
import hashlib, json
from pathlib import Path
manifiesto = json.loads(Path("datos/mensajes_cab/versiones.json").read_text(encoding="utf-8"))
for archivo, info in manifiesto["archivos"].items():
    real = hashlib.sha256(Path("datos/mensajes_cab", archivo).read_bytes()).hexdigest()
    print(f"{archivo:24s} {'OK' if real == info['sha256'] else 'NO COINCIDE'}  {real[:12]}…")
print("Versión de datos:", manifiesto["version"])

## 3. Instalación de dependencias

Se instalan las mismas versiones de `transformers`, `tokenizers` y `sentencepiece` que usa la API dentro del contenedor, para que el modelo se comporte igual en Colab y en Docker.

In [ ]:
!pip install -q -r entrenamiento/requirements-colab.txt
from importlib.metadata import version
for libreria in ["torch", "transformers", "tokenizers", "sentencepiece", "datasets", "scikit-learn"]:
    print(f"{libreria:14s} {version(libreria)}")

## 4. Autenticación en Hugging Face Hub

El token no se escribe en el código: se lee de *Secrets*. Debe tener permiso de escritura porque al final se publica el modelo.

In [ ]:
from huggingface_hub import login, whoami
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception as error:
    print("No se pudo leer HF_TOKEN de Secrets:", type(error).__name__)
USUARIO_HF = whoami()["name"]
REPO_MODELO = f"{USUARIO_HF}/router-intencion-cab-mdeberta"
print("Sesión iniciada como:", USUARIO_HF)
print("El modelo se publicará en:", REPO_MODELO)

## 5. Datos: data-v3.0.0

| Archivo | Contenido | Uso |
|---|---|---|
| `entrenamiento.csv` | 3,200 mensajes (Bitext + CAB simulado) | Ajustar el modelo |
| `validacion.csv` | 400 mensajes | Elegir la mejor época, calibrar la confianza y elegir el umbral |
| `prueba.csv` | 400 mensajes | Medir con mensajes que el modelo no vio, pero del mismo estilo |
| `prueba_externa_v3.csv` | 160 mensajes escritos a mano | Medición honesta: no sale de las plantillas |

Dos cambios respecto a la A4: los 135 mensajes de Bitext que preguntaban por el "ETA" se reetiquetaron como `fecha_entrega` (en CAB el ETA es la fecha estimada), y el número de orden se agregó también a mensajes de otras intenciones para que "traer un número de OC" deje de delatar el estatus (atajo detectado en la A5).

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 100)
entrena = pd.read_csv("datos/mensajes_cab/entrenamiento.csv")
externa = pd.read_csv("datos/mensajes_cab/prueba_externa_v3.csv")
display(pd.crosstab(entrena["intencion"], [entrena["fuente"], entrena["idioma"]], margins=True))
display(pd.crosstab(externa["tipo"], externa["idioma"], margins=True))
display(entrena.groupby("fuente").sample(4, random_state=1)[["mensaje", "intencion", "idioma", "fuente"]])

## 6. Ajuste, calibración y evaluación

El script hace todo el proceso en orden:

1. **Fine-tuning** con la configuración de la A4 (tasa de aprendizaje 2e-5, lotes de 8, 3 épocas, *weight decay* 0.01, modelo convertido a 32 bits). Se agrega *label smoothing* de 0.1: en lugar de enseñarle al modelo que la respuesta correcta vale 100 %, se le enseña que vale 90 %. Así no aprende a estar "seguro de todo", que fue el problema de la A5 (errores con confianza de 0.97).
2. **Calibración con temperatura** en validación: se busca un número T que divide las salidas del modelo para que una confianza de 0.9 signifique, de verdad, acertar cerca de 9 de cada 10 veces.
3. **Umbral:** el menor umbral que en validación logra al menos 98 % de precisión en las respuestas automáticas (KPI del riesgo R05), con un mínimo de 0.90 (Fase 1).
4. **Evaluación final** con el modelo **ya guardado y recargado en CPU**, igual que lo cargará la API dentro del contenedor.

In [ ]:
!python entrenamiento/entrenar_router.py --salida salida_router

## 7. Resultados

**Respondidas en automático** es la proporción de mensajes que el chatbot contestaría solo. **Automáticas erróneas** son los mensajes respondidos en automático con una intención equivocada: es el error que más importa, porque el cliente recibiría información que no pidió. Aclarar, escalar o canalizar no son errores graves: el cliente termina con una persona.

In [ ]:
import json
from IPython.display import Image, display
m = json.loads(Path("salida_router/resultados/metricas_router.json").read_text(encoding="utf-8"))
print(f"Temperatura: {m['config']['temperatura']} | umbral: {m['config']['umbral_confianza']} | "
      f"entrenamiento: {m['config']['segundos_entrenamiento']:.0f} s en {m['config']['entrenado_en']}")
tabla = pd.DataFrame(m["resultados"]).T[["mensajes", "accuracy", "f1_macro", "cobertura_automatica",
                                         "precision_automatica", "resultados_seguros", "respuestas_automaticas_erroneas"]]
display(tabla)
ext = m["resultados"]["externa"]
print("Prueba externa · accuracy por idioma:", ext["accuracy_por_idioma"])
print("Prueba externa · resultados seguros por tipo:", ext["seguros_por_tipo"])
print("Prueba externa · acciones:", ext["acciones"])
print(f"ECE sin calibrar {ext['ece_sin_calibrar']:.3f} → calibrado {ext['ece_calibrado']:.3f}")
print("Latencia en CPU:", {k: round(v, 1) if isinstance(v, float) else v for k, v in m["latencia"].items()})
display(pd.DataFrame(m["historial"]))

In [ ]:
for figura in ["matriz_externa", "matriz_prueba", "calibracion_externa", "umbral_externa"]:
    display(Image(f"salida_router/resultados/figuras/{figura}.png", width=520))

## 8. Análisis de errores en la prueba externa

Se revisan los mensajes mal clasificados y, sobre todo, los que el chatbot habría respondido en automático por error.

In [ ]:
pred = pd.read_csv("salida_router/resultados/predicciones_externa.csv")
columnas = ["mensaje", "intencion", "prediccion", "confianza", "accion", "idioma", "tipo"]
print("Respuestas automáticas erróneas (graves):")
display(pred[~pred["seguro"]][columnas])
print("Errores de intención que terminaron en una acción segura (aclarar / escalar / canalizar):")
display(pred[pred["seguro"] & ~pred["acierto"] & (pred["tipo"] != "ambiguo")][columnas])
print("Mensajes ambiguos y fuera de dominio:")
display(pred[pred["tipo"].isin(["ambiguo", "fuera_dominio"])][columnas])

## 9. Comparación con la Actividad 5

En la A5 el router de regresión logística obtuvo, con 50 mensajes escritos a mano, una exactitud de 0.525 en inglés y de 0.20 en español. La tabla compara esos mismos 50 mensajes (origen `A5`) con el modelo de este cuaderno.

In [ ]:
a5 = pred[pred["origen"] == "A5"]
comparacion = pd.DataFrame({
    "A5 · regresión logística (Bitext)": {"en": 0.525, "es": 0.20},
    "Fase 2 · mDeBERTa-v3 (data-v3.0.0)": a5.groupby("idioma")["acierto"].mean().round(3).to_dict(),
})
display(comparacion)

## 10. Publicación en Hugging Face Hub

Se sube el modelo con su ficha (*model card*), sus métricas y `config_cab.json` (temperatura, umbral, versión de datos y commit de Git), y se crea la etiqueta `model-v1.0.0`. El `Dockerfile` de la API descarga exactamente esa etiqueta.

In [ ]:
!python entrenamiento/publicar_modelo_hf.py --carpeta salida_router --repo {REPO_MODELO}

## 11. Descarga de resultados

Se descarga un `.zip` con las métricas, predicciones y figuras. Se copia a `resultados/router/` del repositorio local para documentar la validación.

In [ ]:
import shutil
!cp salida_router/modelo/config_cab.json salida_router/resultados/
shutil.make_archive("/content/resultados_router", "zip", "salida_router/resultados")
from google.colab import files
files.download("/content/resultados_router.zip")
print("Modelo publicado en: https://huggingface.co/" + REPO_MODELO)